# Program Pengujian: Mengenali 1 Gambar

Alur: *upload* 1 gambar, baca berkas bobot, hitung umpan maju saja, lalu tampilkan apakah gambar dikenali atau tidak.
Mengikuti *Application procedure* Fausett, *Fundamentals of Neural Networks* (1994), hlm. 299 sampai 300.

Jalankan sel dari atas ke bawah. Sel 4 akan meminta Anda memilih 1 gambar.

## Sel 1. Persiapan Lingkungan dan Konfigurasi Pengujian

* Menyambungkan Google Colab ke Google Drive untuk membaca model tersimpan.
* Menentukan alamat berkas model (`BOBOT.npz`), daftar 3 label kelas (`PESAWAT`, `MOBIL`, `KAPAL`), serta ambang keyakinan minimum (*threshold* = 0.5) agar prediksi dianggap sah atau dikenali.
* Menyediakan opsi variabel `KELAS_SEBENARNYA` jika ingin langsung memvalidasi apakah tebakan model cocok dengan kelas asli dari gambar yang diunggah.

In [ ]:
import io
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import drive, files

drive.mount('/content/drive')

BERKAS_BOBOT = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI/BOBOT/BOBOT.npz'
AMBANG_DIKENALI = 0.5    # nilai output terbesar minimal 0,5 agar dianggap dikenali
NAMA_KELAS = ['PESAWAT', 'MOBIL', 'KAPAL']

# Isi bila Anda tahu kelas gambar yang akan diunggah (0 = pesawat, 1 = mobil, 2 = kapal).
# Biarkan None bila tidak ingin dibandingkan.
KELAS_SEBENARNYA = None

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Berkas bobot    : {BERKAS_BOBOT}')
print(f'Ambang dikenali : {AMBANG_DIKENALI}')
print(f'Kelas           : {", ".join(NAMA_KELAS)}')

## Sel 2. Pemuatan Bobot Model Terlatih

* Memeriksa keberadaan berkas `BOBOT.npz` di Google Drive. Jika tidak ada, program menampilkan tombol unggah dari komputer lokal untuk memasukkan berkas bobot secara manual.
* Membaca seluruh larik parameter hasil latihan: bobot dan bias lapisan tersembunyi ($v, v_0$), bobot dan bias lapisan *output* ($w, w_0$), serta dimensi resolusi gambar ($32 \times 32$).
* Melakukan validasi struktur jaringan untuk memastikan jumlah *node input* (1024) dan *node output* (3) sudah sesuai dengan arsitektur model.

In [ ]:
if os.path.exists(BERKAS_BOBOT):
    bobot = np.load(BERKAS_BOBOT)
    sumber = 'Google Drive'
else:
    print('Berkas bobot tidak ada di Drive. Silakan unggah BOBOT.npz.')
    unggahan = files.upload()
    bobot = np.load(io.BytesIO(unggahan[list(unggahan.keys())[0]]))
    sumber = 'unggahan'

v   = bobot['v']
v_0 = bobot['v_0']
w   = bobot['w']
w_0 = bobot['w_0']
LEBAR, TINGGI = [int(a) for a in bobot['ukuran']]

n = v.shape[0]   # node input
p = v.shape[1]   # node hidden
m = w.shape[1]   # node output

print('=' * 52)
print(f'BOBOT DIBACA (dari {sumber})')
print('=' * 52)
print(f'{"Variabel":<10}{"Ukuran":<14}')
for nama, larik in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)]:
    print(f'{nama:<10}{str(larik.shape):<14}')
print(f'\nNode input (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p): {p}')
print(f'Node output (m): {m}')
assert m == 3 and n == LEBAR * TINGGI, 'Ukuran bobot tidak sesuai'

## Sel 3. Fungsi Prapemrosesan dan Umpan Maju (Inferensi)

* Fungsi aktivasi (`f`): menyiapkan fungsi sigmoid untuk memetakan hasil perhitungan ke rentang 0 sampai 1.
* Fungsi `praproses`: mengubah gambar yang diunggah menjadi hitam putih (*grayscale*), menyesuaikan ukuran ke $32 \times 32$, meratakan *pixel* menjadi larik 1 dimensi (1024 nilai), dan menormalisasi nilainya ke rentang 0 sampai 1.
* Fungsi `umpan_maju`: menjalankan inferensi satu arah (murni tebakan maju tanpa koreksi mundur) dari *input* melewati lapisan *hidden* hingga menghasilkan 3 nilai skor di lapisan *output*.

In [ ]:
def f(x):
    return 1.0 / (1.0 + np.exp(-x))


def praproses(gambar):
    """Gambar PIL (RGB) -> vektor x berukuran n, nilai 0 sampai 1."""
    abu = gambar.convert('L')
    if abu.size != (LEBAR, TINGGI):
        abu = abu.resize((LEBAR, TINGGI))
    return np.asarray(abu, dtype=np.float64).flatten() / 255.0


def umpan_maju(x):
    z_in = v_0 + x @ v      # node hidden
    z    = f(z_in)
    y_in = w_0 + z @ w      # node output
    y    = f(y_in)
    return y

print('Fungsi siap: praproses(gambar) dan umpan_maju(x)')

## Sel 4. Pengunggahan Gambar Uji

* Membuka fitur interaktif pemilihan berkas dari komputer lokal untuk memilih 1 gambar uji.
* Membaca berkas gambar yang dipilih, memeriksa format warnanya (RGB), dan menampilkan dimensi resolusi gambar aslinya di layar.

In [ ]:
unggahan = files.upload()
nama_berkas = list(unggahan.keys())[0]
if len(unggahan) > 1:
    print(f'Anda memilih {len(unggahan)} berkas. Yang dipakai hanya yang pertama: {nama_berkas}')

gambar = Image.open(io.BytesIO(unggahan[nama_berkas]))
print(f'Berkas   : {nama_berkas}')
print(f'Ukuran   : {gambar.size[0]} x {gambar.size[1]} pixel')
print(f'Mode     : {gambar.mode}')

## Sel 5. Eksekusi Prediksi dan Visualisasi Hasil

* Menjalankan fungsi prapemrosesan dan perhitungan umpan maju terhadap gambar yang baru saja diunggah.
* Mengambil kelas dengan nilai skor tertinggi:
   * Jika skor tertinggi $\ge 0.5$, model memutuskan gambar DIKENALI sebagai kelas tersebut.
   * Jika skor tertinggi $< 0.5$, model menyatakan gambar TIDAK DIKENALI karena tingkat keyakinan terlalu rendah.
* Menampilkan visualisasi 3 panel: foto asli yang diunggah, citra hasil konversi *grayscale* $32 \times 32$, serta diagram batang skor *output* ($Y_1, Y_2, Y_3$) lengkap dengan garis batas ambang pengenalan.

In [ ]:
x = praproses(gambar)
y = umpan_maju(x)

indeks = int(np.argmax(y))
nilai_maks = float(y[indeks])
dikenali = nilai_maks >= AMBANG_DIKENALI

print('=' * 52)
print('HASIL PENGENALAN')
print('=' * 52)
print(f'{"Node output":<14}{"Kelas":<10}{"Nilai":>8}')
for k in range(m):
    tanda = '  <-- terbesar' if k == indeks else ''
    print(f'Y{k + 1:<13}{NAMA_KELAS[k]:<10}{y[k]:>8.3f}{tanda}')
print('-' * 52)

if dikenali:
    print(f'DIKENALI sebagai {NAMA_KELAS[indeks]} (nilai {nilai_maks:.3f}, ambang {AMBANG_DIKENALI})')
else:
    print(f'TIDAK DIKENALI (nilai terbesar {nilai_maks:.3f}, di bawah ambang {AMBANG_DIKENALI})')

if KELAS_SEBENARNYA is not None:
    cocok = (indeks == KELAS_SEBENARNYA) and dikenali
    print(f'Kelas sebenarnya: {NAMA_KELAS[KELAS_SEBENARNYA]} -> {"BENAR" if cocok else "SALAH"}')

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(11, 3.2))
a1.imshow(gambar); a1.set_title('Gambar asli'); a1.axis('off')
a2.imshow(x.reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1); a2.set_title('Masuk ke jaringan (grayscale)'); a2.axis('off')
warna = ['tab:green' if k == indeks else 'tab:gray' for k in range(m)]
a3.bar([f'Y{k + 1}\n{NAMA_KELAS[k]}' for k in range(m)], y, color=warna)
a3.axhline(AMBANG_DIKENALI, color='tab:red', linestyle='--', linewidth=1)
a3.set_ylim(0, 1); a3.set_title('Nilai output')
plt.tight_layout()
plt.show()